In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata
import time
import os
import wget
from datetime import timedelta
import scgen
from scgen.file_utils import ensure_dir_for_file
sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.settings.set_figure_params(dpi=80)  # low dpi (dots per inch) yields small inline figures
sc.logging.print_versions()

/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
2026-02-17 02:51:48.665712: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-17 02:51:48.764781: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH:

Instructions for updating:
non-resource variables are not supported in the long term


Package,Version
wget,3.2
Component,Info
Python,"3.10.19 | packaged by conda-forge | (main, Jan 26 2026, 23:45:08) [GCC 14.3.0]"
OS,Linux-6.1.159-181.297.amzn2023.x86_64-x86_64-with-glibc2.35
CPU,"16 logical CPU cores, x86_64"
GPU,"ID: 0, NVIDIA L40S, Driver: 580.126.09, Memory: 46068 MiB"
Updated,2026-02-17 02:51
Dependency,Version
pytz,2025.2
flatbuffers,25.12.19


In [2]:
train_path = "../data/pancreas.h5ad"
if os.path.isfile(train_path):
    adata = scgen.load_file(train_path)
else:
    train_url = "https://www.dropbox.com/s/zvmt8oxhfksumw2/pancreas.h5ad?dl=1"
    t_dl = wget.download(train_url, train_path)
    adata = scgen.load_file(train_path)
adata = anndata.AnnData(X=np.expm1(adata.raw.X), var=adata.raw.var, obs=adata.obs)
sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e4)
filter_result = sc.pp.filter_genes_dispersion(
    adata.X, min_mean=0.0125, max_mean=2.5, min_disp=0.7)
adata = adata[:, filter_result.gene_subset]
sc.pp.log1p(adata)

/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/python3.10/site-packages/anndata/compat/__init__.py:371: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(
/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/python3.10/site-packages/anndata/compat/__init__.py:371: FutureWarning: Moving element from .uns['neighbors']['connectivities'] to .obsp['connectivities'].

This is where adjacency matrices should go now.
  warn(


normalizing by total count per cell
    finished (0:00:00): normalized adata.X and added
    'n_counts', counts per cell before normalization (adata.obs)
extracting highly variable genes


/tmp/ipykernel_12819/33765751.py:9: FutureWarning: Use sc.pp.normalize_total instead
  sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e4)
/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/python3.10/site-packages/scanpy/preprocessing/_simple.py:590: FutureWarning: Use sc.pp.normalize_total instead
  normalize_per_cell(
/tmp/ipykernel_12819/33765751.py:10: FutureWarning: Use sc.pp.highly_variable_genes instead
  filter_result = sc.pp.filter_genes_dispersion(


    finished (0:00:00)


/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/python3.10/site-packages/scanpy/preprocessing/_simple.py:412: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


In [3]:
%load_ext rpy2.ipython

In [4]:
%%R
# Install batchelor (provides mnnCorrect) and BiocParallel from R (not conda) to avoid bioconda post-link errors on SageMaker.
# Uses BiocManager (biocLite() is deprecated). Note: mnnCorrect moved from scran to batchelor in Bioconductor.
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager", repos = "https://cloud.r-project.org")
if (!("batchelor" %in% rownames(installed.packages())))
  BiocManager::install("batchelor")
if (!("BiocParallel" %in% rownames(installed.packages())))
  BiocManager::install("BiocParallel")

* installing *source* package ‘BiocManager’ ...
** this is package ‘BiocManager’ version ‘1.30.27’
** package ‘BiocManager’ successfully unpacked and MD5 sums checked
** using staged installation
** R
** inst
** byte-compile and prepare package for lazy loading
** help
*** installing help indices
*** copying figures
** building package indices
** installing vignettes
** testing if installed package can be loaded from temporary location
** testing if installed package can be loaded from final location
** testing if installed package keeps a record of temporary installation path
* DONE (BiocManager)
* installing *source* package ‘formatR’ ...
** this is package ‘formatR’ version ‘1.14’
** package ‘formatR’ successfully unpacked and MD5 sums checked
** using staged installation
** R
** inst
** byte-compile and prepare package for lazy loading
** help
*** installing help indices
** building package indices
** installing vignettes
** testing if installed package can be loaded from temporary

Creating a new generic function for ‘rowAlls’ in package ‘MatrixGenerics’
Creating a new generic function for ‘colAlls’ in package ‘MatrixGenerics’
Creating a new generic function for ‘rowAnyNAs’ in package ‘MatrixGenerics’
Creating a new generic function for ‘colAnyNAs’ in package ‘MatrixGenerics’
Creating a new generic function for ‘rowAnys’ in package ‘MatrixGenerics’
Creating a new generic function for ‘colAnys’ in package ‘MatrixGenerics’
Creating a new generic function for ‘rowAvgsPerColSet’ in package ‘MatrixGenerics’
Creating a new generic function for ‘colAvgsPerRowSet’ in package ‘MatrixGenerics’
Creating a new generic function for ‘rowCollapse’ in package ‘MatrixGenerics’
Creating a new generic function for ‘colCollapse’ in package ‘MatrixGenerics’
Creating a new generic function for ‘rowCounts’ in package ‘MatrixGenerics’
Creating a new generic function for ‘colCounts’ in package ‘MatrixGenerics’
Creating a new generic function for ‘rowCummaxs’ in package ‘MatrixGenerics’
C

** help
*** installing help indices
** building package indices
** testing if installed package can be loaded from temporary location
** testing if installed package can be loaded from final location
** testing if installed package keeps a record of temporary installation path
* DONE (MatrixGenerics)
* installing *source* package ‘assorthead’ ...
** this is package ‘assorthead’ version ‘1.4.0’
** package ‘assorthead’ successfully unpacked and MD5 sums checked
** using staged installation
** inst
** help
*** installing help indices
** building package indices


No man pages found in package  ‘assorthead’ 


** installing vignettes
** testing if installed package can be loaded from temporary location
** testing if installed package can be loaded from final location
** testing if installed package keeps a record of temporary installation path
* DONE (assorthead)
* installing *source* package ‘rsvd’ ...
** this is package ‘rsvd’ version ‘1.0.5’
** package ‘rsvd’ successfully unpacked and MD5 sums checked
** using staged installation
** R
** data
*** moving datasets to lazyload DB
** inst
** byte-compile and prepare package for lazy loading
** help
*** installing help indices
** building package indices
** testing if installed package can be loaded from temporary location
** testing if installed package can be loaded from final location
** testing if installed package keeps a record of temporary installation path
* DONE (rsvd)
* installing *source* package ‘snow’ ...
** this is package ‘snow’ version ‘0.4-4’
** package ‘snow’ successfully unpacked and MD5 sums checked
** using staged installa

...

x86_64-conda-linux-gnu-c++ -std=gnu++17 -I"/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/R/include" -DNDEBUG -I../inst/include -DSTRICT_R_HEADERS -I'/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/R/library/Rcpp/include' -I'/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/R/library/BH/include' -I'/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/R/library/sitmo/include' -DNDEBUG -D_FORTIFY_SOURCE=2 -O2 -isystem /home/sagemaker-user/.conda/envs/scgen-repro-env/include -I/home/sagemaker-user/.conda/envs/scgen-repro-env/include -Wl,-rpath-link,/home/sagemaker-user/.conda/envs/scgen-repro-env/lib    -fpic  -fvisibility-inlines-hidden  -fmessage-length=0 -march=nocona -mtune=haswell -ftree-vectorize -fPIC -fstack-protector-strong -fno-plt -O2 -ffunction-sections -pipe -isystem /home/sagemaker-user/.conda/envs/scgen-repro-env/include -fdebug-prefix-map=/home/conda/feedstock_root/build_artifacts/r-base-split_1766426576771/work=/usr/local/src/conda/r-base-4.5.2 -fdebug-prefi

installing to /home/sagemaker-user/.conda/envs/scgen-repro-env/lib/R/library/00LOCK-dqrng/00new/dqrng/libs
** R
** inst
** byte-compile and prepare package for lazy loading
** help
*** installing help indices
** building package indices
** installing vignettes
** testing if installed package can be loaded from temporary location
** checking absolute paths in shared objects and dynamic libraries
** testing if installed package can be loaded from final location
** testing if installed package keeps a record of temporary installation path
* DONE (dqrng)
* installing *source* package ‘igraph’ ...
** this is package ‘igraph’ version ‘2.2.2’
** package ‘igraph’ successfully unpacked and MD5 sums checked
** using staged installation


libxml2 include directories: -I/opt/conda/include/libxml2 -I/opt/conda/include
libxml2 library link flags: -L/opt/conda/lib -lxml2 -L/opt/conda/lib -lz -L/opt/conda/lib -llzma -L/opt/conda/lib -licui18n -licuuc -licudata -L/opt/conda/lib -liconv -lm -ldl
Using installed GLPK
x86_64-conda-linux-gnu-gfortran -fvisibility=hidden -fpic  -march=nocona -mtune=haswell -ftree-vectorize -fPIC -fstack-protector-strong -fno-plt -O2 -ffunction-sections -pipe -isystem /home/sagemaker-user/.conda/envs/scgen-repro-env/include -fdebug-prefix-map=/home/conda/feedstock_root/build_artifacts/r-base-split_1766426576771/work=/usr/local/src/conda/r-base-4.5.2 -fdebug-prefix-map=/home/sagemaker-user/.conda/envs/scgen-repro-env=/usr/local/src/conda-prefix  -c vendor/arpack/dgetv0.f -o vendor/arpack/dgetv0.o


** libs
using C compiler: ‘x86_64-conda-linux-gnu-cc (conda-forge gcc 13.4.0-16) 13.4.0’
using C++ compiler: ‘x86_64-conda-linux-gnu-c++ (conda-forge gcc 13.4.0-16) 13.4.0’
make: x86_64-conda-linux-gnu-gfortran: No such file or directory
make: *** [/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/R/etc/Makeconf:228: vendor/arpack/dgetv0.o] Error 127
ERROR: compilation failed for package ‘igraph’
* removing ‘/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/R/library/igraph’
* restoring previous ‘/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/R/library/igraph’


trying URL 'https://cloud.r-project.org/src/contrib/BiocManager_1.30.27.tar.gz'
Content type 'application/x-gzip' length 752490 bytes (734 KB)
downloaded 734 KB


The downloaded source packages are in
	‘/tmp/RtmpPh3xzb/downloaded_packages’
Updating HTML index of packages in '.Library'
Making 'packages.html' ... done
Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.2 (2025-10-31)
Installing package(s) 'BiocVersion', 'batchelor'
also installing the dependencies ‘formatR’, ‘lambda.r’, ‘futile.options’, ‘GenomicRanges’, ‘MatrixGenerics’, ‘Biobase’, ‘IRanges’, ‘Seqinfo’, ‘S4Arrays’, ‘assorthead’, ‘rsvd’, ‘XVector’, ‘sparseMatrixStats’, ‘futile.logger’, ‘snow’, ‘SingleCellExperiment’, ‘SummarizedExperiment’, ‘S4Vectors’, ‘BiocGenerics’, ‘BiocNeighbors’, ‘BiocSingular’, ‘SparseArray’, ‘DelayedArray’, ‘DelayedMatrixStats’, ‘BiocParallel’, ‘scuttle’, ‘ResidualMatrix’, ‘ScaledMatrix’, ‘beachmat’

trying URL 'https://cloud.r-project.org/src/contrib/formatR_1.14.tar.gz'
trying URL 'https://c

In [5]:
df1 = pd.DataFrame(data=adata[adata.obs['sample']=='Baron'].X.todense().transpose(),
                  index=adata[adata.obs['sample']=='Baron'].var_names,
                  columns=adata[adata.obs['sample']=='Baron'].obs_names)

df2 = pd.DataFrame(data=adata[adata.obs['sample']=='Muraro'].X.todense().transpose(),
                  index=adata[adata.obs['sample']=='Muraro'].var_names,
                  columns=adata[adata.obs['sample']=='Muraro'].obs_names)

df3 = pd.DataFrame(data=adata[adata.obs['sample']=='Segerstolpe'].X.todense().transpose(),
                  index=adata[adata.obs['sample']=='Segerstolpe'].var_names,
                  columns=adata[adata.obs['sample']=='Segerstolpe'].obs_names)

df4 = pd.DataFrame(data=adata[adata.obs['sample']=='Wang'].X.todense().transpose(),
                  index=adata[adata.obs['sample']=='Wang'].var_names,
                  columns=adata[adata.obs['sample']=='Wang'].obs_names)

In [6]:
%%R -i df1 -i df2 -i df3 -i df4 -o odf1 -o odf2 -o odf3 -o odf4

suppressMessages(library(parallel))   # for detectCores()
suppressMessages(library(batchelor))
suppressMessages(library(BiocParallel))
suppressMessages(library(SingleCellExperiment))   # assay(), colData() for mnnCorrect result

t1 = Sys.time()
mnncount = mnnCorrect(data.matrix(df1), data.matrix(df2), data.matrix(df3), data.matrix(df4), 
                      BPPARAM=MulticoreParam(detectCores()))
t2 = Sys.time()
print(t2-t1)

# batchelor returns a SingleCellExperiment: one "corrected" assay (genes x cells); cells are in same order as inputs
corrected_mat = assay(mnncount, "corrected")
n1 = ncol(df1); n2 = ncol(df2); n3 = ncol(df3); n4 = ncol(df4)
odf1 = as.data.frame(corrected_mat[, 1:n1])
odf2 = as.data.frame(corrected_mat[, (n1+1):(n1+n2)])
odf3 = as.data.frame(corrected_mat[, (n1+n2+1):(n1+n2+n3)])
odf4 = as.data.frame(corrected_mat[, (n1+n2+n3+1):(n1+n2+n3+n4)])

Time difference of 13.34173 mins


In [7]:
adata_mnncorrect = adata.copy()
adata_mnncorrect.X = np.concatenate((odf1.values.T, odf2.values.T, odf3.values.T, odf4.values.T))
sc.pp.scale(adata_mnncorrect, max_value=10)

In [8]:
adata_mnncorrect.write(ensure_dir_for_file("../data/mnn.h5ad"))